# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [10]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [11]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x = TODO  # apply the low-rank update (batch, in_dim) -> (batch, out_dim)
        # return x
        lora_update = (x @ self.A @ self.B) * self.alpha
        return lora_update
# Hyperparameters for the sandbox test
random_seed = 123
# in_dim = TODO
# out_dim = TODO
# rank = TODO
# alpha = TODO
in_dim = 10     # Input features
out_dim = 20    # Output features
rank = 4        # Low-rank bottleneck (r)
alpha = 1.0     # Scaling factor

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
# x = TODO  # e.g., torch.randn(batch, in_dim)
# Create dummy input (batch_size=2, in_dim=10)
x = torch.randn(2, in_dim)
print(x)
print(layer)
print("Original output:", layer(x))

tensor([[-0.2044, -2.2685, -0.9133, -0.4204,  0.2436, -0.0567,  0.3784,  1.6863,
          0.2553, -0.5496],
        [ 1.0042,  0.8272,  1.5434,  0.1406,  1.0617, -0.9929, -1.6025, -1.0764,
          0.9031, -0.7218]])
LoRALayer()
Original output: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
       grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [12]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    # def forward(self, x):
    #     return TODO
    def forward(self, x):
        # The base output (frozen knowledge)
        base_output = self.linear(x)

        # The adaptive output (trainable delta)
        lora_output = self.lora(x)

        # Add them together: y = Wx + dWx
        return base_output + lora_output

base_linear = nn.Linear(in_dim, out_dim)
# layer_lora_1 = TODO  # wrap `base_linear` with rank/alpha values
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[ 0.0772,  0.6381,  0.3749,  0.5645,  0.5885, -1.2343, -0.4951, -0.1743,
          0.2426,  0.5950,  0.1876,  0.8564,  0.4490,  0.2703,  0.1804, -0.1308,
          1.0212, -0.7650,  0.0305,  0.1894],
        [ 0.5396, -0.4899, -0.9066,  0.2121, -0.1434,  0.6912,  0.5024, -0.1358,
         -1.0635, -0.7070, -0.0855, -0.4941,  0.2096,  0.2800,  0.4152, -0.4292,
         -0.7979,  0.1503,  0.9157,  1.3572]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [13]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

# single_net = SingleLayerNet(num_features=TODO, num_classes=TODO)
# sample_input = TODO
single_net = SingleLayerNet(num_features=10, num_classes=3)
# sample_input = TODO
sample_input = torch.randn(1, 10)

with torch.no_grad():
    baseline_output = single_net(sample_input)

# single_net.layer = TODO  # replace with LinearWithLoRA
single_net.layer = LinearWithLoRA(single_net.layer, rank=4, alpha=8)

with torch.no_grad():
    lora_output = single_net(sample_input)

# print("Outputs match before training?", TODO)
# Using torch.allclose to handle tiny floating point variances
match = torch.allclose(baseline_output, lora_output, atol=1e-6)
print("Outputs match before training?", match)

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [14]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B
        # combined_weight = TODO  # merge original weights + scaled LoRA correction
        combined_weight = self.linear.weight + (lora.t() * self.lora.alpha)
        return F.linear(x, combined_weight, self.linear.bias)

# layer_lora_2 = TODO
layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=rank, alpha=alpha)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[ 0.0772,  0.6381,  0.3749,  0.5645,  0.5885, -1.2343, -0.4951, -0.1743,
          0.2426,  0.5950,  0.1876,  0.8564,  0.4490,  0.2703,  0.1804, -0.1308,
          1.0212, -0.7650,  0.0305,  0.1894],
        [ 0.5396, -0.4899, -0.9066,  0.2121, -0.1434,  0.6912,  0.5024, -0.1358,
         -1.0635, -0.7070, -0.0855, -0.4941,  0.2096,  0.2800,  0.4152, -0.4292,
         -0.7979,  0.1503,  0.9157,  1.3572]], grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [23]:
# class MultilayerPerceptron(nn.Module):
#     def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
#         super().__init__()
#         self.layers = nn.Sequential(
#             TODO,
#             nn.ReLU(),
#             TODO,
#             nn.ReLU(),
#             TODO,
#         )
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            # First Hidden Layer
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            # Second Hidden Layer
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            # Output Layer (No ReLU here so we can use CrossEntropyLoss)
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.layers(x)
        return x

In [24]:
# Architecture
# num_features = TODO
# num_hidden_1 = TODO
# num_hidden_2 = TODO
# num_classes = TODO
num_features=784
num_hidden_1=128
num_hidden_2=64
num_classes=10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# learning_rate = TODO
# num_epochs = TODO
learning_rate = 0.005
num_epochs = 1

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
# optimizer_pretrained = TODO
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

cuda
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [25]:
BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

# test_dataset = TODO
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor())

# train_loader = TODO
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# test_loader = TODO
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [26]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            # features = TODO
            # targets = TODO
            # logits = TODO
# Move data to the specified device (GPU)
            features = features.to(device)
            targets = targets.to(device)

            # Forward pass
            logits = model(features)

            _, predicted_labels = torch.max(logits, 1)
    #         num_examples += TODO
    #         correct_pred += TODO
    # return TODO
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()

    return correct_pred / num_examples * 100

## Training

In [27]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            # features = TODO
            # targets = TODO

            # logits = TODO
            # loss = TODO
            features = features.to(device)
            targets = targets.to(device)

            # Forward pass: compute predicted outputs by passing inputs to the model
            logits = model(features)

            # Calculate loss
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [28]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

/tmp/ipykernel_605/3407308978.py:26: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/001|Batch 000/938| Loss: 2.3023
Epoch: 001/001|Batch 400/938| Loss: 0.2224
Epoch: 001/001|Batch 800/938| Loss: 0.2166
Epoch: 001/001 training accuracy: 96.68%
Time elapsed: 0.26 min
Total Training Time: 0.26 min
Test accuracy: 95.98%


# Replacing Linear with LoRA Layers

In [30]:
model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
# model_lora.layers[2] = TODO
# model_lora.layers[4] = TODO
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model:95.98%
Test accuracy LoRA model:95.98%


## Freezing the Original Linear Layers

In [31]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.0.linear.weight:False
layers.0.linear.bias:False
layers.0.lora.A:True
layers.0.lora.B:True
layers.2.linear.weight:False
layers.2.linear.bias:False
layers.2.lora.A:True
layers.2.lora.B:True
layers.4.linear.weight:False
layers.4.linear.bias:False
layers.4.lora.A:True
layers.4.lora.B:True


In [32]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Epoch: 001/001|Batch 000/938| Loss: 0.1174
Epoch: 001/001|Batch 400/938| Loss: 0.0380
Epoch: 001/001|Batch 800/938| Loss: 0.0803
Epoch: 001/001 training accuracy: 96.54%
Time elapsed: 0.24 min
Total Training Time: 0.24 min
Test accuracy LoRA finetune: 95.83%
Test accuracy orig model:95.98%
Test accuracy LoRA model:95.83%
